#Web Search

In [2]:
!pip install -qU langchain-google-genai google-generativeai tavily-python

In [3]:
user_query = "What is the capital of India?"

In [4]:
from google.colab import userdata
from tavily import TavilyClient

def web_search(query):

    client = TavilyClient(userdata.get('TAVILY'))

    response = client.search(
        query=query,
        search_depth="advanced",
        max_results=5
    )

    contents = []

    for r in response["results"]:
        if "content" in r:
            contents.append(r["content"])

    research_content = "\n\n".join(contents)

    return research_content


web_results = web_search(user_query)

#LLM Creation

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=userdata.get('GEMINI_API_KEY')
)

response = llm.invoke("Hi, who are you?")
print(response.content)

I am a large language model, trained by Google.


#Imports For LangChain

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

#Summary

In [8]:
summary_prompt = PromptTemplate(
    template="""
You are an expert research analyst.

Summarize the following research into 4–5 clear factual lines.

Rules:
- Easy to understand
- No fluff
- No opinions
- Serve as a reliable source of truth

Research:
{research}

Summary:
""",
    input_variables=["research"]
)

summary_chain = summary_prompt | llm | StrOutputParser()

#FaceBook Chain

In [9]:
fb_prompt = PromptTemplate(
    template="""
You are an expert social media manager working at a FAANG-level company.

Create a short Facebook caption.

Rules:
- Under 150 characters
- Catchy and fun
- Emojis allowed
- Optional hashtags
- Caption style

Topic Summary:
{summary}

Facebook Caption:
""",
    input_variables=["summary"]
)

fb_chain = fb_prompt | llm | StrOutputParser()

#LinkedIn Chain

In [10]:
linkedin_prompt = PromptTemplate(
    template="""
You are a senior social media strategist.

Create a professional LinkedIn post.

Rules:
- Professional and thoughtful
- First-person perspective
- 2–4 short paragraphs
- Value-driven
- Avoid hashtags unless needed

Topic Summary:
{summary}

LinkedIn Post:
""",
    input_variables=["summary"]
)

linkedin_chain = linkedin_prompt | llm | StrOutputParser()

#Twitter Chain

In [11]:
twitter_prompt = PromptTemplate(
    template="""
Write a Twitter/X post.

Rules:
- Under 280 characters
- Punchy and engaging
- Conversational tone
- Emojis allowed
- 2–3 hashtags

Topic Summary:
{summary}

Tweet:
""",
    input_variables=["summary"]
)

twitter_chain = twitter_prompt | llm | StrOutputParser()

#RunnableParallel

In [12]:
from langchain_core.runnables import RunnableParallel

Parallel_Chain = RunnableParallel(
    linkedin=linkedin_chain,
    twitter=twitter_chain,
    facebook=fb_chain
)

#End-to-End Pipeline Function

In [13]:
def generate_posts(query: str):

    print(f" Searching for: {query}")

    research_content = web_search(query)

    print(" Generating summary...")

    summary = summary_chain.invoke({
        "research": research_content
    })

    print(" Generating posts in parallel...")

    posts = Parallel_Chain.invoke({
        "summary": summary
    })

    result = {
        "query": query,
        "summary": summary,
        "linkedin": posts["linkedin"],
        "twitter": posts["twitter"],
        "facebook": posts["facebook"]
    }

    return result

#Display Function

In [14]:
def display_results(result):

    print("\n" + "="*60)
    print("QUERY")
    print("="*60)
    print(result["query"])

    print("\n" + "="*60)
    print("SUMMARY")
    print("="*60)
    print(result["summary"])

    print("\n" + "="*60)
    print("LINKEDIN POST")
    print("="*60)
    print(result["linkedin"])

    print("\n" + "="*60)
    print("TWITTER/X POST")
    print("="*60)
    print(result["twitter"])

    print("\n" + "="*60)
    print("FACEBOOK CAPTION")
    print("="*60)
    print(result["facebook"])

#Save JSON + MarkDown

In [15]:
import json

def save_results(result, filename="posts"):

    # Save JSON
    with open(f"{filename}.json", "w") as f:
        json.dump(result, f, indent=4)

    md_content = f"""
# Web2Post Output

## Query
{result['query']}

## Summary
{result['summary']}

## LinkedIn
{result['linkedin']}

## Twitter
{result['twitter']}

## Facebook
{result['facebook']}
"""

    with open(f"{filename}.md", "w") as f:
        f.write(md_content)

    print(f"\n Saved: {filename}.json and {filename}.md")

#Sample Run 1 (Tech Topic)

In [17]:
result1 = generate_posts("Latest AI regulations 2026")

display_results(result1)

save_results(result1, "posts_ai")

 Searching for: Latest AI regulations 2026
 Generating summary...
 Generating posts in parallel...

QUERY
Latest AI regulations 2026

SUMMARY
1.  Multiple US states, including Colorado, California, and Texas, are enacting significant AI legislation, with many laws taking effect in 2026.
2.  These state laws impose varied requirements, such as a duty of care to prevent algorithmic discrimination (Colorado), disclosures for companion chatbots (California), and prohibitions on certain AI uses (Texas).
3.  The federal AI regulatory environment is in flux, with a new executive order proposing to preempt inconsistent state laws.
4.  Businesses face increased regulatory scrutiny, with the SEC identifying AI-driven threats as a FY2026 examination priority and considering enhanced disclosure requirements.
5.  Cyber insurance carriers are increasingly requiring AI-specific security controls, like red-teaming and model-level risk assessments, for coverage.

LINKEDIN POST
The AI regulatory landsca

#Sample Run 2 (Sports Topic)

In [18]:
result2 = generate_posts("Asia Cup cricket 2025")

display_results(result2)

save_results(result2, "posts_sports")

 Searching for: Asia Cup cricket 2025
 Generating summary...
 Generating posts in parallel...

QUERY
Asia Cup cricket 2025

SUMMARY
Here is a summary of the 2025 Men's Asia Cup:

*   The 2025 Men's Asia Cup, the 17th edition, took place in the United Arab Emirates from September 9 to 28, 2025.
*   The tournament featured eight teams and was played in the Twenty20 International (T20I) format.
*   India defeated Pakistan by 5 wickets in the final, securing their ninth title and successfully defending their 2023 championship.
*   Abhishek Sharma was awarded Player of the Series and scored the most runs (314), while Kuldeep Yadav took the most wickets (17).

LINKEDIN POST
The recent conclusion of the 2025 Men's Asia Cup in the UAE offered more than just thrilling cricket; it was a masterclass in sustained performance and captivating narrative. Witnessing India successfully defend their title against Pakistan in a nail-biting T20I final truly underscored the power of consistency and strateg